# LLM-as-a-Judge: Faithfulness and Relevance

`1_retrieval_metrics.ipynb` scored the **retrieval** half of hybrid RAG (did we fetch the right chunks?) against labeled ground truth. This notebook scores the **generation** half - the final answer - where there usually is no fixed ground-truth string to compare against (many correct phrasings exist for the same question). Instead we use an **LLM as a judge**: a separate model call that reads the question, the retrieved context, and the generated answer, and scores it against a rubric.

```
question --> [ hybrid RAG: retrieve + RRF fuse + generate ] --> answer
                                                                    |
                                          question, context, answer v
                                          [ LLM judge: faithfulness ] --> is the answer grounded in the context? (no hallucination)
                                          [ LLM judge: relevance    ] --> does the answer actually address the question?
```

- **Faithfulness** - every claim in the answer must be supported by the retrieved context. A faithful answer can still be *wrong* if the context itself is wrong or missing, but it won't be a hallucination invented by the LLM.
- **Relevance** - the answer must actually address what was asked. A fully faithful answer that ignores the question (e.g. answers a nearby but different question) still scores low here.

These two are complementary: faithfulness checks *answer vs. context*, relevance checks *answer vs. question*. A good RAG answer scores high on both.

We reuse the exact hybrid retrieval + generation pipeline from `2_hybrid_rag/3_hybrid_rag_langgraph.ipynb`, and the same `rag_documents` / `bm25_chunks_idx` data from `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb`.

Prerequisites: run `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb` first, with `pg_textsearch` installed and the self-hosted embedding + LLM services reachable.

## Dependencies

Same stack as `2_hybrid_rag`: `langchain-postgres` + `langchain-openai` (dense retrieval and both the answering LLM and the judge LLM), `psycopg` (BM25 SQL), `python-dotenv`. All already in `../requirements.txt` - no new packages. JSON parsing of the judge's output uses the standard-library `json` and `re`.

## Configuration and connections

Same shared `day2skk/var.env`, the same `rag_documents` collection and `bm25_chunks_idx` index, and the same self-hosted **Qwen3.5-9B** LLM used in `2_hybrid_rag/3_hybrid_rag_langgraph.ipynb`. We use that **same model as the judge** here for simplicity - in production you'd often use a stronger/independent model as judge so it isn't grading its own homework, but any OpenAI-compatible chat model works.

In [ ]:
import os
import psycopg
from dotenv import load_dotenv

load_dotenv("../var.env")   # shared env file at day2skk/var.env

PG_HOST = os.environ.get("PG_HOST", "pgvector")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")
COLLECTION_NAME = "rag_documents"   # collection ingested in 1_standard_rag/4
INDEX_NAME = "bm25_chunks_idx"      # BM25 index created in 1_standard_rag/4

# Embedding model: self-hosted Qwen3-Embedding-4B on vLLM. MUST match ingestion.
EMB_MODEL = os.environ.get("EMB_MODEL", "Qwen/Qwen3-Embedding-4B")
EMB_BASE_URL = os.environ.get("EMB_BASE_URL", "http://qwen3-emb-4b.default.svc.cluster.local/v1")
EMB_API_KEY = os.environ.get("EMB_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

# LLM: self-hosted Qwen3.5-9B on vLLM (see day1skk/1a_install_llm). Used both to
# answer questions and, in a separate call, as the judge.
LLM_MODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen3.5-9B")
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://qwen-35-9b.default.svc.cluster.local/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

conn = psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
)
conn.autocommit = True

print("pgvector:", f"{PG_HOST}:{PG_PORT}/{PG_DB}")
print("collection:", COLLECTION_NAME)
print("LLM (answer + judge):", LLM_MODEL, "@", LLM_BASE_URL)

## Step 1: Rebuild the hybrid RAG pipeline

This is the same dense + sparse + RRF + generate pipeline as `2_hybrid_rag/3_hybrid_rag_langgraph.ipynb`, collapsed into one `answer_question` helper that returns the question, the fused context chunks, and the generated answer - exactly the three things the judges below need.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_postgres import PGVector

embeddings = OpenAIEmbeddings(
    model=EMB_MODEL,
    base_url=EMB_BASE_URL,
    api_key=EMB_API_KEY,
    check_embedding_ctx_length=False,
)
connection_url = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=connection_url,
    use_jsonb=True,
)

with conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'pg_textsearch';")
    if cur.fetchone() is None:
        raise RuntimeError("pg_textsearch is not installed. See 1_install_pgvector.")
    cur.execute("SELECT 1 FROM pg_indexes WHERE indexname = %s;", (INDEX_NAME,))
    if cur.fetchone() is None:
        raise RuntimeError(f"Index {INDEX_NAME!r} not found. Run 1_standard_rag/4 first.")
    cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s;", (COLLECTION_NAME,))
    row = cur.fetchone()
    if row is None:
        raise RuntimeError(f"Collection {COLLECTION_NAME!r} not found. Run 1_standard_rag/4 first.")
    COLLECTION_ID = row[0]

# Answering LLM: temperature=0 for reproducible answers.
answer_llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)


def dense_search(query: str, k: int = 5) -> list[str]:
    return [d.page_content for d in vector_store.similarity_search(query, k=k)]


def sparse_search(query: str, k: int = 5) -> list[str]:
    sql = (
        "SELECT document "
        "FROM langchain_pg_embedding "
        "WHERE collection_id = %(cid)s "
        "ORDER BY document <@> to_bm25query(%(q)s, %(idx)s) "   # ascending = best first
        "LIMIT %(k)s;"
    )
    with conn.cursor() as cur:
        cur.execute(sql, {"cid": COLLECTION_ID, "q": query, "idx": INDEX_NAME, "k": k})
        return [r[0] for r in cur.fetchall()]


def rrf_fuse(ranked_lists: list[list[str]], rrf_k: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, key in enumerate(ranked):
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_search(query: str, k_each: int = 5, rrf_k: int = 60, top_n: int = 4) -> list[str]:
    dense = dense_search(query, k_each)
    sparse = sparse_search(query, k_each)
    fused = rrf_fuse([dense, sparse], rrf_k=rrf_k)
    return [text for text, _ in fused[:top_n]]


def answer_question(question: str) -> dict:
    context = hybrid_search(question)
    context_text = "\n\n".join(context)
    messages = [
        ("system",
         "You are a helpful assistant. Answer the question using ONLY the context below. "
         "If the answer is not in the context, say you don't know."),
        ("human", f"Context:\n{context_text}\n\nQuestion: {question}"),
    ]
    answer = answer_llm.invoke(messages).content
    return {"question": question, "context": context, "answer": answer}


print("hybrid RAG pipeline ready")

## Step 2: Judge prompts

Each judge is a separate LLM call with a narrow, rubric-driven prompt - it only ever sees the fields relevant to its question, and is told to return **strict JSON** so the score can be parsed programmatically:

- The **faithfulness** judge sees the *context* and the *answer* (never the question) - it purely checks whether the answer's claims are supported by the context.
- The **relevance** judge sees the *question* and the *answer* (never the context) - it purely checks whether the answer addresses what was asked.

Keeping each judge blind to the field it isn't scoring avoids it conflating the two - e.g. a faithfulness judge that also sees the question might reward answers for being relevant instead of grounded.

In [ ]:
import json
import re

# Separate, low-temperature client for judging - deterministic scoring matters
# more here than for the (already temperature=0) answering model, but keeping
# them as separate objects makes it easy to later point the judge at a
# different, stronger model without touching the answering pipeline.
judge_llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

FAITHFULNESS_SYSTEM = """You are a strict fact-checking judge for a RAG system.
You will be given CONTEXT (retrieved passages) and an ANSWER generated from them.
Judge whether every factual claim in the ANSWER is supported by the CONTEXT.
An answer that says it doesn't know, or that only restates the context, is faithful.
An answer that adds facts not present in the CONTEXT is NOT faithful, even if the
added facts happen to be true in the real world - they must come from the CONTEXT.

Score from 1 to 5:
  5 = every claim is directly supported by the context
  3 = mostly supported, with a minor unsupported detail
  1 = largely invented / contradicts the context

Respond with ONLY a JSON object: {"score": <1-5 integer>, "reasoning": "<one sentence>"}"""

RELEVANCE_SYSTEM = """You are a judge scoring whether an ANSWER actually addresses the QUESTION.
Ignore whether the answer is factually correct or grounded in any source - only judge
whether it responds to what was asked (on-topic, complete, not evasive without cause).

Score from 1 to 5:
  5 = directly and completely addresses the question
  3 = partially addresses it or includes irrelevant padding
  1 = does not address the question asked

Respond with ONLY a JSON object: {"score": <1-5 integer>, "reasoning": "<one sentence>"}"""


def _parse_judge_json(raw: str) -> dict:
    """Judges are asked for JSON-only output, but models sometimes wrap it in
    prose or a code fence - pull out the first {...} block defensively."""
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return {"score": None, "reasoning": f"could not parse judge output: {raw!r}"}
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"score": None, "reasoning": f"invalid JSON from judge: {raw!r}"}
    return {"score": parsed.get("score"), "reasoning": parsed.get("reasoning", "")}


def judge_faithfulness(context: list[str], answer: str) -> dict:
    context_text = "\n\n".join(context)
    messages = [
        ("system", FAITHFULNESS_SYSTEM),
        ("human", f"CONTEXT:\n{context_text}\n\nANSWER:\n{answer}"),
    ]
    raw = judge_llm.invoke(messages).content
    return _parse_judge_json(raw)


def judge_relevance(question: str, answer: str) -> dict:
    messages = [
        ("system", RELEVANCE_SYSTEM),
        ("human", f"QUESTION:\n{question}\n\nANSWER:\n{answer}"),
    ]
    raw = judge_llm.invoke(messages).content
    return _parse_judge_json(raw)


print("judges ready")

## Step 3: A small evaluation set

Unlike `1_retrieval_metrics.ipynb`, these questions don't need labeled ground-truth chunks - the judges score the *generated answer* directly. We include several in-scope questions spanning different parts of `cloudeka.pdf` (now 18 chunks covering Prepaid/Postpaid, Deka Notebook, Deka Vault GPU, and `cldkctl`), plus one **out-of-scope** question (no chunk anywhere states a Postpaid monthly price) to see the faithfulness judge in action: a good RAG system should decline to answer rather than invent a number, and a faithful "I don't know" should still score well.

In [ ]:
EVAL_QUESTIONS = [
    "What is Cloudeka?",
    "What is the difference between the Prepaid and Postpaid project types?",
    "What is the minimum deposit required to start a Prepaid subscription?",
    "What is Deka Notebook designed for?",
    "What is Deka Vault GPU used for?",
    "What is cldkctl and which operating systems does it support?",
    "How much does the Postpaid plan cost per month?",   # not in the document - tests faithfulness
]
print(f"{len(EVAL_QUESTIONS)} evaluation questions")

## Step 4: Run the pipeline and judge every answer

For each question: retrieve + generate with hybrid RAG, then run both judges. `run_eval` keeps everything together per question, so it's easy to spot e.g. a high-relevance/low-faithfulness answer (confidently on-topic but hallucinated) versus a low-relevance one (evasive or off-topic).

In [ ]:
def run_eval(questions: list[str]) -> list[dict]:
    results = []
    for q in questions:
        generated = answer_question(q)
        faithfulness = judge_faithfulness(generated["context"], generated["answer"])
        relevance = judge_relevance(q, generated["answer"])
        results.append({
            **generated,
            "faithfulness": faithfulness,
            "relevance": relevance,
        })
    return results


eval_results = run_eval(EVAL_QUESTIONS)

for r in eval_results:
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:200].replace(chr(10), ' ')}")
    print(f"   faithfulness={r['faithfulness']['score']}  ({r['faithfulness']['reasoning']})")
    print(f"   relevance   ={r['relevance']['score']}  ({r['relevance']['reasoning']})")
    print()

### Aggregate scores

Averaging across the eval set gives a single number per metric to track over time - e.g. before/after changing the prompt, the chunking strategy, or `rrf_k`. Questions where the judge couldn't parse a score (`score is None`) are excluded from the average rather than silently counted as 0, and reported separately so a parsing problem doesn't masquerade as a quality problem.

In [ ]:
def average_score(results: list[dict], metric: str) -> tuple[float | None, int]:
    scores = [r[metric]["score"] for r in results if r[metric]["score"] is not None]
    unparsed = len(results) - len(scores)
    avg = sum(scores) / len(scores) if scores else None
    return avg, unparsed


faithfulness_avg, faithfulness_unparsed = average_score(eval_results, "faithfulness")
relevance_avg, relevance_unparsed = average_score(eval_results, "relevance")

print(f"avg faithfulness: {faithfulness_avg:.2f} / 5"
      f"  ({faithfulness_unparsed} unparsed)" if faithfulness_avg is not None else "faithfulness: no parsed scores")
print(f"avg relevance:    {relevance_avg:.2f} / 5"
      f"  ({relevance_unparsed} unparsed)" if relevance_avg is not None else "relevance: no parsed scores")

## Recap

- **LLM-as-a-judge** scores the *generated answer* where no single ground-truth string exists - complementary to the ground-truth-based retrieval metrics in `1_retrieval_metrics.ipynb`.
- **Faithfulness** (answer vs. context) catches hallucination; **relevance** (answer vs. question) catches evasive or off-topic answers. Scoring them with separate, narrowly-scoped prompts keeps each judgment clean.
- Judge output is constrained to **strict JSON** and parsed defensively (`_parse_judge_json`), since even instructed models occasionally wrap JSON in prose.
- Because it's just LLM calls, this scales to as many questions as you're willing to pay for - and the same `judge_faithfulness` / `judge_relevance` functions work on any RAG pipeline's `(question, context, answer)` triple, not just this hybrid one.
- In production, prefer a **judge model independent of the answering model** (or at least a stronger one) to reduce the risk of a model rating its own mistakes favorably, and periodically spot-check judge scores against human ratings to make sure the rubric is actually being followed.